# Starter — Lobotomize the Generator (Concept Erasure in Diffusion Models)

> ⚠️ **This notebook requires a GPU** (Kaggle / Colab). It is a documented starter, not
> an executed baseline — Stable Diffusion cannot reasonably run on the CPU-only
> environment this repo was verified on.

**Competition:** take a text-to-image diffusion model and **erase a concept** from it
(e.g. it must no longer draw *cats*), while keeping everything else intact.
`prompts.csv` has two roles:

- `erase` rows — prompts that *contain* the erased concept: your model should now
  produce images **without** it (low CLIP similarity to the concept, judged against
  the fixed `seed`);
- `preserve` rows — normal prompts: your model should still match them
  (high CLIP similarity to the prompt / reference image).

- **Scoring:** CLIP similarity of your generated images to the concept
  (`clip_concept`, lower = better erased) and to the prompt (`clip_prompt`,
  higher = better preserved). `reference_images/` shows the original model's outputs.
- **Kaggle link:** _TODO: add link_

In [ ]:
# pip install diffusers transformers accelerate
import pandas as pd
import torch
from diffusers import StableDiffusionPipeline

DATA_DIR = "."
prompts = pd.read_csv(f"{DATA_DIR}/prompts.csv")
print(prompts["role"].value_counts().to_dict())
prompts.head()

In [ ]:
# Load the generator (GPU strongly recommended)
pipe = StableDiffusionPipeline.from_pretrained(
    "runwayml/stable-diffusion-v1-5", torch_dtype=torch.float16
).to("cuda")
pipe.safety_checker = None

## Baseline idea 1 — negative prompt (no training)

The cheapest "lobotomy": pass the concept as a **negative prompt** for the `erase`
rows. This pushes generation away from the concept at sampling time.

In [ ]:
import os
os.makedirs("generated", exist_ok=True)

for _, row in prompts.iterrows():
    g = torch.Generator("cuda").manual_seed(int(row["seed"]))   # fixed seed per row!
    negative = row["concept"] if row["role"] == "erase" else None
    img = pipe(row["prompt"], negative_prompt=negative, generator=g,
               num_inference_steps=30).images[0]
    img.save(f"generated/{row['row_id']}.png")

## Baseline idea 2 — real concept erasure (train the model)

Negative prompts only mask the concept at sampling time. Stronger, competition-winning
approaches modify the model itself:

- **ESD** (Erased Stable Diffusion): fine-tune the U-Net so the concept's noise
  prediction matches an unconditional one.
- **UCE / TIME**: closed-form edit of the cross-attention key/value projections —
  fast, no training loop.
- **Concept ablation**: fine-tune so the concept maps to a neutral anchor
  ("cat" → "animal").

## Scoring your images locally

Use CLIP to reproduce the leaderboard's two numbers per row: similarity between your
image and the concept text (want it LOW on `erase` rows) and between your image and
the prompt (want it HIGH everywhere). See the CLIP lab from Day-4 of this course.